Nos hemos enfocado anteriormente en ajustar el modelo de regresión lineal utilizando la biblioteca de SKlearn, sin embargo la biblioteca statsmodels ofrece otra alternativa la cual ofrece funciones y reportes adicionales a los que podemos encontrar con las funciones de sklearn

In [1]:
import numpy as np
import pandas as pd
import statsmodels.stats as sms
import statsmodels.api as sm
from scipy.stats import norm, kurtosis

utilizaremos nuevamente el conjunto de datos de publicidad

In [2]:
df = pd.read_csv("publicidad.csv")

In [3]:
df.head(2)

,id,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4


In [4]:
X=df.drop(columns=["id","sales"])
X=sm.add_constant(X) ## se agrega una constante para el intercepto
y=df["sales"]

In [5]:
model=sm.OLS(y,X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     570.3
Date:                Thu, 06 Jun 2024   Prob (F-statistic):           1.58e-96
Time:                        19:17:36   Log-Likelihood:                -386.18
No. Observations:                 200   AIC:                             780.4
Df Residuals:                     196   BIC:                             793.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9389      0.312      9.422      0.0

vemos como el metodo summary nos ofrece un resumen del modelo de regresión. Con este tenemos acceso directo a los coeficientes estimados, así como un intervalo de confianza para cada uno de ellos

Por ejemplo, tenemos que el valor estimado para la variable de televisión es de .0458, sin embargo se nos ofrece un intervalo en el que podemos ver que el valor puede variar entre .043 y .049

Tenemos tambien acceso al resultado de la prueba t para cada una de las variables predictoras, esto nos dice que tan significativas son para el modelo. Si fijamos un nivel de significancia del .05 buscamos que el valor P>|t| sea menor a .05, si sucede esto entonces la variable es significativa, en caso contrario no lo es. Para este conjunto de datos la variable newspaper no es significativa para el modelo.

Tambien se muestra el valor del estadistico durbin watson para ver si los errores no están correlacionados.
Tambien se muestran los valores de AIC (criterio de información de akaike) y BIC (Criterio bayesiano) los cuales tambien nos ayudan en la selección de modelos. 

La idea subyacente en el AIC es encontrar un equilibrio entre la capacidad del modelo para explicar los datos (ajuste) y la penalización por la complejidad del modelo. En otras palabras, se busca un modelo que logre un buen ajuste a los datos pero que no sea excesivamente complejo.

La fórmula del AIC es la siguiente:

AIC = -2 * log(L) + 2 * k

donde:


L es la función de verosimilitud del modelo (una medida de cuán bien el modelo explica los datos),

k es el número de parámetros en el modelo.

El AIC penaliza los modelos más complejos al incluir el término 2⋅k, lo que significa que a medida que aumenta el número de parámetros, el AIC aumenta. Por lo tanto, se prefiere un modelo con un valor de AIC más bajo, ya que indica un buen equilibrio entre ajuste y complejidad.








### Ajustemos un modelo sin la variable newspaper

In [6]:
df = pd.read_csv("publicidad.csv")
X=df.drop(columns=["id","sales","newspaper"])
X=sm.add_constant(X) ## se agrega una constante para el intercepto
y=df["sales"]
model=sm.OLS(y,X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     859.6
Date:                Thu, 06 Jun 2024   Prob (F-statistic):           4.83e-98
Time:                        19:17:36   Log-Likelihood:                -386.20
No. Observations:                 200   AIC:                             778.4
Df Residuals:                     197   BIC:                             788.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9211      0.294      9.919      0.0

Si bien en ambos modelos tenemos los mismos valores de r2, el segundo modelo tiene un valor de AIC más bajo. Por lo que en este caso , el segundo modelo es una mejor opción que el primero

Podemos utilizar Pickle para guardar nuestro modelo ya entrenado y solamente importar al momento de querer hacer predicciones

In [9]:
X.to_csv("columnas_entrenamiento_modeloreglog.csv",index=False)

In [11]:
import pickle

In [12]:
### la sintaxis para guardar el modelo es la siguiente
filename = 'regresion_publicidad.sav'
#pickle.dump(model, open(filename, 'wb'))

In [13]:
### la sintaxis para cargar el modelo es la siguiente
loaded_model = pickle.load(open(filename, 'rb'))

In [17]:
loaded_model.predict([1,230,20]) ## hacemos predicciones con el modelo cargado desde pickle

array([17.20459192])

In [12]:
loaded_model.predict([1,44,39])## hacemos predicciones con el modelo cargado desde pickle

array([12.26608662])

In [13]:
2.9211 + 44*(.0458) + 39*(.1880) # comparamos con la predicción manual

12.2683